# IAN 620 — LangChain Tools + LangGraph + Local Qwen

**Working Colab approach:** This notebook loads Qwen directly using **Hugging Face Transformers**, just like the original [LangChain_LangGraph_Tutorial.ipynb](https://colab.research.google.com/github/ali2449/IAN-620/blob/main/LangChain_LangGraph_Tutorial.ipynb). **Ollama is not required.**

A user asks a question. Qwen sees the LangChain tool descriptions and argument schemas, decides which tool(s) to call, and LangGraph runs those calls.

- **Weather:** API line commented out; returns mock weather.
- **Jobs:** API line commented out; returns fictional job openings.
- **Profit:** calculates real profit using numeric arguments chosen by the model.
- **Other questions:** Qwen answers without calling a tool.

No keyword routing or manually selected agent. Qwen emits native `<tool_call>` messages; a short adapter converts them to LangChain's `AIMessage.tool_calls`, which LangGraph's `ToolNode` can execute.

**Important:** Weather and job results are fictional demonstrations, **not current facts**. For best speed, select **Runtime → Change runtime type → T4 GPU**. Run cells in order.

## 1. Install libraries

This matches the working tutorial's simple Colab setup: pip installs Python libraries, with **no separate model server or Linux installer**. You can execute on CPU, although a GPU is faster.

In [ ]:
%pip -q install -U langchain langgraph transformers accelerate torch

## 2. Load a local Qwen model directly from Hugging Face

The original working notebook used `AutoTokenizer` and `AutoModelForCausalLM`. We use the same approach, with **Qwen3-1.7B**, an instruction model capable of emitting tool calls using its chat template.

The Hugging Face pipeline in the first notebook was suitable for plain text generation. Here we call the tokenizer and model directly so that the **tool schemas are passed to Qwen's chat template**. This is required for native model-driven tool selection. No Ollama or paid API key.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
qwen_model.eval()

print("Qwen ready:", MODEL_ID)
print("Device:", next(qwen_model.parameters()).device)

## 3. Define LangChain tools using `@tool` and docstrings

LangChain automatically turns typed function parameters and docstrings into tool schemas. Qwen reads these **descriptions** when deciding whether to use a tool. The tool definitions are not a routing lookup table.

The weather and jobs functions have commented-out API request examples. For now they return explicitly labeled sample responses.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather information for a city.

    Use for requests about current conditions, temperature, rain, or forecasts.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires a weather API provider and key):
    # response = requests.get(WEATHER_API_URL, params={"city": city, "key": API_KEY}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return f"DEMO WEATHER, not live data: {city}: 72 F, partly cloudy."


@tool
def search_jobs(city: str) -> str:
    """Find recent job opportunities in a specified city.

    Use for requests about hiring, openings, employment, or available jobs.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires an authorized jobs API provider):
    # response = requests.get(JOBS_API_URL, params={"location": city}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"FICTIONAL JOB EXAMPLES ONLY — not verified openings in {city}:\n"
        "1. Data Analyst — Example Analytics Co.\n"
        "2. Python Developer — Demo Software Inc.\n"
        "3. AI Engineer — Sample Technology Group"
    )


@tool
def calculate_profit(selling_price: float, cost_price: float, quantity: int) -> str:
    """Calculate total profit and profit margin for products.

    Use when asked to compute business profit or earnings from selling price,
    cost per item, and quantity. Pass the actual numeric inputs provided by
    the user; if a required value is missing, ask for it instead of guessing.
    """
    if quantity < 0:
        return "Error: quantity must not be negative."
    revenue = selling_price * quantity
    cost = cost_price * quantity
    profit = revenue - cost
    margin = (profit / revenue * 100) if revenue else 0
    return (
        f"Revenue: ${revenue:,.2f}; Total cost: ${cost:,.2f}; "
        f"Profit: ${profit:,.2f}; Margin: {margin:.1f}%."
    )


tools = [get_weather, search_jobs, calculate_profit]

for item in tools:
    print(item.name, "—", item.description.splitlines()[0])

## 4. Give the LangChain tool schemas to Qwen

LangChain's `convert_to_openai_tool` generates schemas from our **tool docstrings**. Qwen's Hugging Face chat template accepts these schemas through `tools=...`.

The model can emit zero, one, or several `<tool_call>` blocks. The adapter below parses those calls into LangChain `AIMessage.tool_calls` so LangGraph can execute them.

**Why not `llm.bind_tools` here?** The original notebook uses a plain Hugging Face generation pipeline, not a chat-model service that reliably implements that interface. Passing tools directly to Qwen's native chat template preserves the original working local setup and still lets the LLM decide tool use.

In [ ]:
import json
import re
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

tool_schemas = [convert_to_openai_tool(t) for t in tools]

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Decide whether a tool is needed using its description and the user's question.
Call all relevant tools when the user requests multiple kinds of information.
Never invent required tool arguments. Ask for missing values.
Weather and jobs tools return sample data, never live information.
Use the profit tool to do arithmetic. After receiving tool results, answer
the question instead of calling the same tool again without reason."""

def to_qwen_message(message):
    """Convert LangChain messages into Qwen's native chat-message format."""
    if isinstance(message, SystemMessage):
        return {"role": "system", "content": message.content}
    if isinstance(message, HumanMessage):
        return {"role": "user", "content": message.content}
    if isinstance(message, ToolMessage):
        return {"role": "tool", "content": str(message.content)}
    if isinstance(message, AIMessage):
        if message.tool_calls:
            calls = [
                {
                    "type": "function",
                    "function": {"name": call["name"], "arguments": call["args"]},
                }
                for call in message.tool_calls
            ]
            return {"role": "assistant", "tool_calls": calls, "content": message.content or ""}
        return {"role": "assistant", "content": str(message.content)}
    raise TypeError(f"Unsupported message: {type(message)}")

def generate_qwen(messages):
    """Ask Qwen to respond, with real tool schemas included in its prompt."""
    chat = [to_qwen_message(m) for m in messages]
    inputs = tokenizer.apply_chat_template(
        chat,
        tools=tool_schemas,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to(qwen_model.device)

    with torch.no_grad():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Qwen's actual model-produced tool calls, not keyword-based routing.
    calls = []
    for match in re.finditer(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated, re.S):
        try:
            item = json.loads(match.group(1))
            if item.get("name") in {t.name for t in tools}:
                args = item.get("arguments", {})
                if isinstance(args, str):
                    args = json.loads(args)
                if isinstance(args, dict):
                    calls.append({"name": item["name"], "args": args, "id": str(uuid4())})
        except (ValueError, TypeError, KeyError):
            pass

    visible_text = re.sub(r"<tool_call>.*?</tool_call>", "", generated, flags=re.S).strip()
    if "<tool_call>" in generated and not calls:
        visible_text = "I could not read the model's tool request. Please try again."

    return AIMessage(content=visible_text, tool_calls=calls)

preview = generate_qwen([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="What is the temperature in Greensboro?"),
])
print("Model-chosen tool calls:", preview.tool_calls)
print("Model text:", preview.content)

## 5. Build LangGraph with `ToolNode`

This is a standard tool-execution loop:

`START → Qwen → ToolNode → Qwen → END`

The LLM chooses tools based on descriptions. `tools_condition` only checks whether the actual LLM message contains tool calls; it does not inspect question strings. `ToolNode` executes any chosen LangChain tools.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    response = generate_qwen([
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("llm", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition, {"tools": "tools", "__end__": END})
builder.add_edge("tools", "llm")

assistant = builder.compile()
print("Multi-tool LangGraph ready.")

## 6. Ask questions and inspect which tools were called

The model decides tool selection. You can inspect the actual calls instead of assuming a fixed route. We set a graph recursion limit to avoid endlessly repeated model/tool steps.

In [ ]:
def ask(question: str) -> None:
    print("\nUSER:", question)
    result = assistant.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 12},
    )

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("LLM SELECTED TOOL:", call["name"])
                print("ARGUMENTS:", call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.content)
    print("FINAL ANSWER:", result["messages"][-1].content)
    print("-" * 60)

ask("How warm is it in Greensboro?")

## 7. Three example requests

The **jobs and weather results are fictional/sample outputs**, while the profit result uses real arithmetic. Try rephrasing the requests to see whether the model picks tools based on their descriptions.

In [ ]:
ask("Are any companies hiring in Greensboro?")
ask("I buy 10 mugs for 50 dollars each and sell them for 75 each. What is my total profit?")

## 8. One question, multiple tools

This request requires two different sources. Qwen should propose tool calls for both. In the printed trace, look for **both** `search_jobs` and `get_weather`.

Note: Small models can still make incorrect selections. Tool-call accuracy is a model capability, not something the docstring guarantees.

In [ ]:
ask("Are there any jobs in Greensboro, and what is the weather there?")

## 9. Try your own question

There are no manual routing rules. Change the `question` variable and run the cell.

In [ ]:
question = "I sell 4 items for $120 each after buying each for $85. What do I earn?"
ask(question)

## 10. Optional: display the graph

This graph illustrates the model → tools → model loop. It is **not** a hardcoded weather/jobs/profit decision tree.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(assistant.get_graph().draw_mermaid_png()))
except Exception:
    print(assistant.get_graph().draw_mermaid())

## 11. Key concepts and next steps

- **Hugging Face Transformers:** loads Qwen directly, like the original working notebook. No Ollama installer.
- **LangChain `@tool`:** the Python function's docstring and type hints define the tool description and argument schema.
- **Qwen's chat template:** passes those schemas to the LLM. Qwen chooses tools by generating native `<tool_call>` blocks.
- **Adapter:** turns Qwen's tool-call markup into LangChain `AIMessage.tool_calls`. It does **not** choose the tools.
- **LangGraph `ToolNode`:** runs the tools and provides results to Qwen.
- **`tools_condition`:** checks for model-generated tool calls and returns to the LLM, or ends.

Model-generated tool calls are not guaranteed to be correct; the 1.7B model may occasionally omit a tool or generate malformed arguments. Check the trace printed by `ask()`. A larger model may improve accuracy.

To connect real services later, uncomment and complete the API lines in the tool functions, handle errors, and protect credentials using Colab Secrets.

References: [Hugging Face tool use](https://huggingface.co/docs/transformers/main/chat_extras) · [Qwen3 tool calling](https://github.com/QwenLM/Qwen3/blob/main/docs/source/framework/function_call.md) · [LangGraph ToolNode](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode)